In [17]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

path = kagglehub.dataset_download("abdullah0a/comprehensive-weight-change-prediction")

df = pd.read_csv(f"{path}/weight_change_dataset.csv")
df.head(5)

,Participant ID,Age,Gender,Current Weight (lbs),BMR (Calories),Daily Calories Consumed,Daily Caloric Surplus/Deficit,Weight Change (lbs),Duration (weeks),Physical Activity Level,Sleep Quality,Stress Level,Final Weight (lbs)
0,1,56,M,228.4,3102.3,3916.0,813.7,0.2,1,Sedentary,Excellent,6,228.6
1,2,46,F,165.4,2275.5,3823.0,1547.5,2.4,6,Very Active,Excellent,6,167.8
2,3,32,F,142.8,2119.4,2785.4,666.0,1.4,7,Sedentary,Good,3,144.2
3,4,25,F,145.5,2181.3,2587.3,406.0,0.8,8,Sedentary,Fair,2,146.3
4,5,38,M,155.5,2463.8,3312.8,849.0,2.0,10,Lightly Active,Good,1,157.5


In [18]:
X = df.drop(["Participant ID","Final Weight (lbs)","Weight Change (lbs)","Daily Caloric Surplus/Deficit"], axis=1)
y = df["Final Weight (lbs)"]
X.head(2)

,Age,Gender,Current Weight (lbs),BMR (Calories),Daily Calories Consumed,Duration (weeks),Physical Activity Level,Sleep Quality,Stress Level
0,56,M,228.4,3102.3,3916.0,1,Sedentary,Excellent,6
1,46,F,165.4,2275.5,3823.0,6,Very Active,Excellent,6


In [19]:
categorical_features = ["Gender","Physical Activity Level","Sleep Quality"]
#                        [0,1],          [2,3,4,5],           [6,7,8,9]
one_hot = OneHotEncoder()
transformer = ColumnTransformer([("one_hot", one_hot,
                                  categorical_features)], remainder="passthrough")

transformed_X = transformer.fit_transform(X)
transformed_X = pd.DataFrame(transformed_X)
transformed_X.head(2)



,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,56.0,228.4,3102.3,3916.0,1.0,6.0
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,46.0,165.4,2275.5,3823.0,6.0,6.0


In [21]:
model = RandomForestRegressor()
np.random.seed(42)
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)
model.fit(X_train,y_train)

model.score(X_test,y_test)

0.9536951657652722

In [24]:
me = [21,"M",158.7,1714,2514,8,'Very Active','Good',3]

In [26]:
# 1. Turn your list into a DataFrame with the original column names
columns = ["Age", "Gender", "Current Weight (lbs)", "BMR (Calories)", 
           "Daily Calories Consumed","Duration (weeks)", 
           "Physical Activity Level", "Sleep Quality", "Stress Level"]

me_df = pd.DataFrame([me], columns=columns)

# 2. Use the transformer you already built to transform 'me_df'
# Note: Use .transform(), NOT .fit_transform()!
me_transformed = transformer.transform(me_df)

# 3. Make the prediction
my_pred_lbs = model.predict(me_transformed)[0]

# Convert to kg
my_pred_kg = my_pred_lbs * 0.45359237

print(f"Predicted Final Weight: {my_pred_kg:.2f} kg")

Predicted Final Weight: 67.16 kg


In [16]:
# 1. Get the feature importances from your Random Forest model
importances = model.feature_importances_

# 2. Create the leaderboard using your column names
# (If 'all_features' gives an error, use X_train.columns instead)
importance_series = pd.Series(importances, index=X_train.columns)

# 3. Sort them from highest to lowest
print("--- Random Forest: Feature Importance Leaderboard ---")
print(importance_series.sort_values(ascending=False))

--- Random Forest: Feature Importance Leaderboard ---
11    0.888912
12    0.048312
16    0.015924
9     0.014859
15    0.009550
10    0.005881
14    0.005178
3     0.003823
13    0.002251
7     0.001682
8     0.001015
0     0.000534
4     0.000504
5     0.000476
2     0.000473
1     0.000357
6     0.000270
dtype: float64
